# دفتر تدقيق وتنظيف بيانات الوظائف التقنية

**تاريخ التنفيذ:** 2026-10-28<br>
**مرجع البيانات:** `unit-test`<br>
**الملف المنظف:** `/tmp/tmp4cdtj2ns/processed/jobs.csv.gz`

## النطاق

شُغّل الـpipeline على 1 مصادر متاحة، بإجمالي 3 صفًا. الملفات الخام للقراءة فقط، ويحتفظ `source_row_json` بصف المصدر الأصلي كاملًا. لم تُحذف التكرارات أو القيم الشاذة.

**مصادر مفقودة:** `International_Tech_Jobs_Filtered.csv`, `Merged_Tech_Jobs_45k_skills_enriched_AFTER_ADDING_SKILLS.xlsx`, `arab_tech_jobs_skills_enriched_AFTER_ADDING_SKILLS_439.csv`, `cybersecurity_jobs_expanded_6_columns.csv`, `startup_jobs_foreign_after_adding_skills.csv`, `tech_jobs_jooble_careerjet_AGNABY_after_adding_skills.csv`. هذه نتيجة جزئية ولا تشمل كل مصادر الأرشيف المحددة.

## سياسة التنظيف

- تطبيع المسافات والأسماء ذات المرادف الصريح فقط؛ لا دمج تخميني لعناوين الوظائف أو الدول.
- تحليل تواريخ النشر من أعمدة النشر فقط. حقل `updated` لا يُعامل كتاريخ نشر.
- الحفاظ على نص الراتب وقيم المصدر والعملة والفترة ومراجعها. التقديرات الموروثة موسومة وغير متحققة ومستبعدة من إحصاءات الرواتب الفعلية.
- لا تقدير جديد ولا ملء للراتب المفقود. لا تحويل FX أو سنونة للفترات غير السنوية قبل اعتماد سياسة العملة وسعر الصرف وعوامل السنونة.
- الإحصاءات الفعلية مجمعة حسب المسمى والدولة/النطاق والعملة والفترة؛ متوسط/وسيط القيمة يمثل منتصف النطاق المعلن.

## نتائج التنفيذ

- الصفوف قبل/بعد: **3 / 3**؛ المحذوف **0**.
- تكرارات دقيقة محفوظة ومعلّمة: **1**.
- رواتب `actual_reported`: **3**؛ منها المؤهل لإحصاءات حسب العملة والفترة: **2**.
- تقديرات موروثة غير متحققة: **0**؛ لا تُخلط مع الفعلي.
- الرواتب المفقودة التي بقيت مفقودة: **0**.
- قاعدة بيانات المشروع: **not_available_in_current_checkout**؛ اختبار SQLite بالذاكرة: **passed**.

## التشغيل من جديد

بعد توفير جميع ملفات المصدر في مجلد واحد، شغّل:

```bash
python scripts/data_cleaning/build_cleaning_audit.py \
  --raw-data-directory data/raw \
  --source-ref "main@<commit>"
```

استخدم ملف سياسة صريحًا فقط بعد اعتماده، عبر `--salary-policy-file`; من دون ذلك تبقى تحويلات العملة والسنونة غير مطبقة.


## 1. تحميل النتائج وجرد المدخلات

الخلايا التالية تستخدم Python القياسي فقط. يبدأ التدقيق بمرجع النسخة وmanifest والبصمات؛ الملفات الخام تُقرأ ولا تُعدّل.

In [ ]:
from pathlib import Path
import csv
import gzip
import json


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "data_cleaning" / "build_cleaning_audit.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the notebook working directory")


PROJECT_ROOT = find_project_root()
REPORT_DIR = PROJECT_ROOT / "analysis" / "reports" / "data_cleaning"
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "jobs_cleaned_audited.csv.gz"
SUMMARY = json.loads((REPORT_DIR / "cleaning_summary.json").read_text(encoding="utf-8"))
BEFORE = json.loads((REPORT_DIR / "data_quality_before.json").read_text(encoding="utf-8"))
AFTER = json.loads((REPORT_DIR / "data_quality_after.json").read_text(encoding="utf-8"))


def read_csv_rows(path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


print("scope:", SUMMARY["scope"])
print("rows before / after:", SUMMARY["input_rows"], SUMMARY["output_rows"])
print("dataset:", DATASET_PATH)


In [ ]:
manifest = read_csv_rows(REPORT_DIR / "source_manifest.csv")
print("Source manifest (including missing files and base-export references)")
for item in manifest:
    print(item["status"], item["role"], item["file"], item["sha256"][:12])

print("\nBase/enriched skill comparisons")
for filename, result in SUMMARY["base_enriched_pair_audit"].items():
    print(filename, result["status"], result["matched_ids_with_changed_skills"], "changed IDs")


## 2. مثال على صف خام ومشتقاته

يعرض هذا المثال حقولًا محددة فقط، ويقارن القيمة الأصلية بالعمود المشتق. `source_row_json` يحفظ كامل صف المصدر، أما `title_key` فيطابق النص بعد casefold/مسافات فقط ولا يدمج الأدوار دلاليًا.

In [ ]:
with gzip.open(DATASET_PATH, "rt", encoding="utf-8", newline="") as handle:
    example = next(csv.DictReader(handle))
source_row = json.loads(example["source_row_json"])
print("Raw source values retained:", {
    "title": source_row.get("title", source_row.get("job_title", "")),
    "country": source_row.get("country", source_row.get("job_country", "")),
    "salary": source_row.get("salary", source_row.get("salary_raw", "")),
})
print("Derived fields:", {
    "title_clean": example["title_clean"],
    "country_standardized": example["country_standardized"],
    "posted_at_iso": example["posted_at_iso"],
    "source_updated_at_iso": example["source_updated_at_iso"],
    "skills_tokens_json": example["skills_tokens_json"],
    "salary_currency_code": example["salary_currency_code"],
    "salary_period_code": example["salary_period_code"],
    "salary_value_type": example["salary_value_type"],
})


## 3. جودة البيانات قبل التنظيف وبعده

المفقود يبقى مفقودًا؛ تُعرض أيضًا التواريخ القديمة/المستقبلية، النصوص المشكوك بترميزها، والتكرارات كإشارات مراجعة فقط.

In [ ]:
print("Before-cleaning field coverage")
for field, values in BEFORE["field_coverage"].items():
    print(f"{field:30} present={values['present']:>8} missing={values['missing']:>8}")

print("\nAfter-cleaning field coverage")
for field, values in AFTER["field_coverage"].items():
    print(f"{field:30} present={values['present']:>8} missing={values['missing']:>8}")

print("\nCleaning invariants")
print("rows dropped:", SUMMARY["rows_dropped"])
print("exact duplicate occurrences retained:", SUMMARY["exact_duplicate_occurrences_retained"])
print("raw source files unchanged:", SUMMARY["source_integrity"]["all_unchanged_during_run"])


## 4. الرواتب والتحليل المعاد تشغيله

لا تستخدم متوسطًا موحدًا بين عملات أو فترات مختلفة. الإحصاءات تقتصر على مبالغ `actual_reported` ذات العملة والفترة الصريحتين، بعد حجب القيم التي تحتاج مراجعة واسعة. التقديرات القديمة غير المتحققة منفصلة، ولا تُستخدم لتعبئة المفقود.

In [ ]:
from itertools import islice

salary_stats_path = REPORT_DIR / "salary_statistics_actual.csv"
with gzip.open(DATASET_PATH, "rt", encoding="utf-8", newline="") as handle:
    cleaned_reader = csv.DictReader(handle)
    cleaned_sample = list(islice(cleaned_reader, 5))
    cleaned_row_count = len(cleaned_sample) + sum(1 for _ in cleaned_reader)

print("cleaned rows:", cleaned_row_count)
print("salary value-type counts:", SUMMARY["salary"]["value_type_counts"])
print("actual-statistics-eligible rows:", SUMMARY["salary"]["actual_statistics_eligible_records"])
print("currency policy:", SUMMARY["salary_policy"]["status"])
print("FX conversion applied:", SUMMARY["salary_policy"].get("fx_conversion_applied", False))
print("non-year annualization policy:", SUMMARY["salary_policy"].get("annualization_factors", {}))

salary_stats = read_csv_rows(salary_stats_path)
print("actual salary groups:", len(salary_stats))
for row in salary_stats[:10]:
    print({key: row[key] for key in (
        "country", "title_display", "salary_currency_code", "salary_period_code",
        "actual_record_count", "average_of_record_midpoints_native", "median_of_record_midpoints_native"
    )})


## 5. اختبارات الاحتفاظ والاتساق

التكرار أو الشذوذ يوسم للمراجعة ولا يؤدي للحذف. اختبار SQLite داخل الذاكرة يراجع اتساق CSV والإحصاءات فقط؛ لا يحل محل المقارنة بقاعدة المشروع غير الموجودة.

In [ ]:
# Every source row is retained; duplicate and outlier flags are review metadata, not deletion rules.
assert SUMMARY["input_rows"] == SUMMARY["output_rows"]
assert SUMMARY["rows_dropped"] == 0
assert cleaned_row_count == SUMMARY["output_rows"]
assert SUMMARY["sqlite_round_trip"]["status"] == "passed"

# The old project database is checked separately from this in-memory round-trip.
print("existing database:", SUMMARY["existing_database_status"])
print("in-memory SQLite salary-group check:", SUMMARY["sqlite_round_trip"])
